# 02 · Реализовать геометрию потока и сравнить два метода
К лекции 9. Шесть TODO: warping, resize векторов, masked EPE, LK system, forward/backward consistency и собственные контрпримеры. Затем Farneback и DIS на одинаковых парах с известным движением; val → фиксированный test и абляция разрешения.

Время работы студента: 150–190 минут. Вычисления — секунды/минуты на CPU; тяжелого обучения и весов нет. Пакеты: numpy, matplotlib, opencv-python. Для реальной ветки при первом запуске скачивается ~8.1 МБ видео в temp cache с SHA; синтетическая часть полностью локальна.

**Соглашение:** f₁₂(x) на сетке I1 переносит x в x+f₁₂(x) кадра I2. x вправо, y вниз; единицы — исходные пиксели между кадрами. Î₁(x)=I₂(x+f₁₂(x)). Backward flow задан на сетке I2: для проверки нужно прочитать его в x+f₁₂(x), а не просто сложить массивы.

GT-valid — совместно видимые исходные пиксели согласно генератору. Одна и та же маска используется у всех методов. Consistency mask — только диагностика; она не заменяет GT mask.

In [ ]:
# Запускайте первой. В Colab сохраняем штатные PyTorch/CUDA и ipywidgets.
import importlib.util, subprocess, sys
if sys.version_info < (3, 10):
    raise RuntimeError('Нужен Python 3.10 или новее.')
_packages = {'numpy': 'numpy>=1.26,<3', 'matplotlib': 'matplotlib>=3.8,<4', 'cv2': 'opencv-python-headless==4.12.0.88'}
_missing = [spec for module, spec in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
    raise RuntimeError('Пакеты установлены. Перезапустите среду выполнения и выполните ноутбук сверху.')
import numpy, matplotlib
if not (1 <= int(numpy.__version__.split('.')[0]) < 3):
    raise RuntimeError('Нужен NumPy 1.x или 2.x; используйте отдельное окружение.')
if int(matplotlib.__version__.split('.')[0]) != 3:
    raise RuntimeError('Нужен Matplotlib 3.x; используйте отдельное окружение.')
_versions = {'python': sys.version.split()[0], 'numpy': numpy.__version__,
             'matplotlib': matplotlib.__version__}
if 'scipy' in _packages:
    import scipy
    if int(scipy.__version__.split('.')[0]) != 1:
        raise RuntimeError('Нужен SciPy 1.x; используйте отдельное окружение.')
    _versions['scipy'] = scipy.__version__
if 'cv2' in _packages:
    import cv2
    _versions['opencv'] = cv2.__version__
    _required_features = ('calcOpticalFlowFarneback', 'DISOpticalFlow_create')
    _absent = [name for name in _required_features if not hasattr(cv2, name)]
    if _absent:
        raise RuntimeError('В OpenCV нет ' + ', '.join(_absent) +
                           '. Установите один вариант OpenCV 4.12.0.88 в отдельной среде и перезапустите ядро.')
if 'ipywidgets' in _packages:
    import ipywidgets
    _versions['ipywidgets'] = ipywidgets.__version__
    if int(ipywidgets.__version__.split('.')[0]) not in (7, 8):
        raise RuntimeError('Нужен ipywidgets 7.x или 8.x; в Colab 2026.07 оставьте штатный 7.7.1.')
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
print('Окружение:', _versions)
print('Вычисления этой лабораторной: CPU; GPU не требуется.')

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import time,json,csv,hashlib,os
cv2.setNumThreads(1)
plt.rcParams.update({'figure.dpi':110,'font.size':10})
assert hasattr(cv2,'calcOpticalFlowFarneback') and hasattr(cv2,'DISOpticalFlow_create')
print('OpenCV',cv2.__version__,'; CPU threads:',cv2.getNumThreads())

In [ ]:
def grid(h,w):
 y,x=np.mgrid[:h,:w];return x.astype(np.float32),y.astype(np.float32)

def texture(seed,h=128,w=192):
 rng=np.random.default_rng(seed);a=cv2.GaussianBlur(rng.normal(0,1,(h,w)).astype(np.float32),(0,0),.8)
 a=(a-a.mean())/(a.std()+1e-8)*35+127
 a=np.uint8(np.clip(a,0,255));a[10:35,12:52]=128
 return a

def flow_rgb(flow,vmax):
 u,v=flow[...,0].astype(np.float32),flow[...,1].astype(np.float32)
 mag,angle=cv2.cartToPolar(u,v)
 hsv=np.empty((*u.shape,3),np.uint8);hsv[...,0]=np.uint8(angle/(2*np.pi)*179);hsv[...,1]=255;hsv[...,2]=np.uint8(np.clip(mag/vmax,0,1)*255)
 return cv2.cvtColor(hsv,cv2.COLOR_HSV2RGB)

def draw_legend():
 x,y=grid(81,81);f=np.dstack([(x-40)/40,(y-40)/40]);rgb=flow_rgb(f,1);rgb[np.linalg.norm(f,axis=2)>1]=255
 fig,ax=plt.subplots(figsize=(3,3));ax.imshow(rgb);ax.set(xticks=[0,40,80],xticklabels=['−u','0','+u'],yticks=[0,40,80],yticklabels=['−v','0','+v'],title='Direction; brightness = magnitude');plt.show();plt.close(fig)

In [ ]:
draw_legend()

## Гипотеза и протокол до запуска
**Заполните:** какой метод ожидаете лучше на чистом переносе и на границе движущейся вставки; как половинное разрешение изменит время и ошибку после правильного возврата векторов в исходные пиксели.

Val: четыре seeds и четыре заранее заданных смещения, каждый в двух сценах (общий перенос, движущаяся вставка). Test: четыре другие seeds и смещения, тоже две сцены. Параметры Farneback и DIS заранее фиксированы. Выбираем метод с минимальной pooled GT-valid val EPE; tie → порядок списка методов. После выбора создается test.

На test сравниваем оба полноразмерных метода и выбранный метод при scale .5. Единственное изменение абляции — разрешение входа. Flow возвращается в исходную сетку с масштабированием u/v. Основная EPE всегда в исходных пикселях. Нельзя подбирать параметры, masks или seeds по test.

Это контролируемая синтетическая задача с точной геометрией, а не результат на Sintel/KITTI. Реальное видео в конце не имеет используемой здесь GT.

## TODO 1 · Реконструкция с forward flow
Постройте карты grid + flow и маску выхода за границу. Функция должна работать также для двухканального поля, чтобы затем семплировать backward flow.

In [ ]:
def warp_with_forward(image2,flow12):
    """Same HxW images; flow shape HxWx2, finite. Image2 may be gray or channels.
    Return image2 sampled at grid + flow12, and boolean in-bounds mask.
    Use cv2.remap, linear interpolation, constant-zero border. Float32 output.
    Valid coordinates: 0<=x<=W-1 and 0<=y<=H-1. Reject nonfinite/shape mismatch."""
    raise NotImplementedError('TODO 1: warp with forward field')

In [ ]:
a=np.arange(20,dtype=np.float32).reshape(4,5);f=np.zeros((4,5,2),np.float32);f[...,0]=1
w,m=warp_with_forward(a,f);assert np.array_equal(w[:,:-1],a[:,1:]) and not m[:,-1].any() and m[:,:-1].all()
w0,m0=warp_with_forward(a,np.zeros_like(f));assert np.array_equal(w0,a) and m0.all()
try:warp_with_forward(a,np.full_like(f,np.nan))
except ValueError:pass
else:raise AssertionError('nonfinite flow')
print('Warp tests passed')

## TODO 2 · Сетка и единицы векторов
Интерполируйте поле, затем независимо масштабируйте u по ширине и v по высоте. Не меняйте исходный массив.

In [ ]:
def resize_flow(flow,new_shape):
    """Resize HxWx2 flow to (new_H,new_W), then scale u by new_W/W,
    v by new_H/H. Linear interpolation, no mutation. Finite nonempty field
    and positive integer new dimensions required."""
    raise NotImplementedError('TODO 2: resize grid and vector components')

In [ ]:
f=np.zeros((4,6,2),np.float32);f[:]=[3,-2];before=f.copy();r=resize_flow(f,(8,3))
assert r.shape==(8,3,2) and np.allclose(r,[1.5,-4]) and np.array_equal(f,before)
assert np.allclose(resize_flow(r,(4,6)),f)
try:resize_flow(f,(0,3))
except ValueError:pass
else:raise AssertionError('invalid output dimensions')
print('Resize tests passed')

## TODO 3 · Только общая GT-valid область
Посчитайте среднюю норму ошибки и полную карту с NaN на invalid. Пустая valid область не является нулевой ошибкой.

In [ ]:
def masked_epe(pred,truth,valid):
    """Return dict mean, count, map (NaN outside valid). Euclidean endpoint error.
    Validate matching HxWx2 fields and HxW mask. Nonfinite on valid => ValueError;
    nonfinite outside valid is ignored. Zero valid => mean NaN, count0."""
    raise NotImplementedError('TODO 3: masked EPE')

In [ ]:
gt=np.zeros((2,2,2));gt[:]=[2,1];pred=gt+[-.3,.4];mask=np.array([[1,1],[1,0]],bool);pred[1,1]=[100,100]
r=masked_epe(pred,gt,mask);assert np.isclose(r['mean'],.5) and r['count']==3 and np.isnan(r['map'][1,1])
r=masked_epe(pred,gt,np.zeros((2,2),bool));assert np.isnan(r['mean']) and r['count']==0
pred[0,0]=np.nan
try:masked_epe(pred,gt,mask)
except ValueError:pass
else:raise AssertionError('NaN in valid region')
print('EPE tests passed')

## TODO 4 · Решение и наблюдаемость
Решите локальную линейную систему. Надежность определяйте по матрице данных до добавления ridge: регуляризатор не создает недостающий градиент.

In [ ]:
def solve_lk(A,It,ridge=0.):
    """A has rows [Ix,Iy], It shape N. Normalize normal equations by N.
    Solve (A.T@A/N + ridge*I) f = -A.T@It/N with lstsq, no explicit inverse.
    Return flow, eigenvalues of UNREGULARIZED A.T@A/N, observable=min_eigen>1e-6.
    Validate N>0, shapes, finite inputs and finite ridge>=0."""
    raise NotImplementedError('TODO 4: LK and observability')

In [ ]:
r=solve_lk([[1,0],[0,1],[1,1]],[-2,-1,-3]);assert np.allclose(r['flow'],[2,1]) and r['observable']
r=solve_lk([[1,0],[1,0]],[-2,-2]);assert np.allclose(r['flow'],[2,0]) and not r['observable']
r=solve_lk([[1,0],[1,0]],[-2,-2],.1);assert not r['observable']
print('LK tests passed')

## TODO 5 · Обратное поле живет на другой сетке
Семплируйте f21 в точках x+f12(x). Проверка affine case ниже специально отличается от постоянного переноса и обнаруживает сложение массивов без переноса координат.

In [ ]:
def forward_backward_check(flow12,flow21,threshold):
    """Same-sized grids. Sample flow21 at x+flow12(x), using warp_with_forward.
    Residual norm(flow12 + sampled_backward), NaN outside bounds.
    Return residual, inside, consistent=inside & residual<=threshold.
    Reject different shapes or negative/nonfinite threshold.
    Do NOT add the two arrays at unchanged pixel coordinates."""
    raise NotImplementedError('TODO 5: forward/backward consistency')

In [ ]:
x,y=grid(3,9);f=np.dstack([.5*x,np.zeros_like(x)]);b=np.dstack([-x/3,np.zeros_like(x)])
res,inside,ok=forward_backward_check(f,b,1e-5)
assert inside[:,:6].all() and not inside[:,6:].any() and ok[inside].all()
assert np.nanmax(res)<1e-5 and np.max(np.linalg.norm(f+b,axis=2))>1
print('Consistency tests passed: reverse field sampled on the other grid')

## TODO 6 · Свои проверки
Два новых случая с ручным прогнозом, кодом assert и объяснением, какую неверную реализацию они отличают. Не копируйте открытые тесты.

In [ ]:
# TODO 6. Create two new cases with manually predicted results and assert.
# One must distinguish grid+flow from grid-flow or proper backward sampling;
# another must test anisotropic scaling, valid-mask handling or aperture rank.
raise NotImplementedError('TODO 6: own counterexamples')

## Синтетические пары и аудит GT
В общем переносе движется вся текстура. Во второй сцене прямоугольная текстурная вставка движется над неподвижным фоном. Пиксели фона, закрытые вставкой во втором кадре, исключены из valid; исходные пиксели вставки остаются допустимыми, если их назначения внутри кадра.

Генератор хранит истинный forward flow, valid, маску границы и слабой текстуры. Проверяется точная реконструкция I1 из I2 по GT на совместно видимых пикселях. Это тест знака, маски и геометрии данных до сравнения алгоритмов.

In [ ]:
H,W=128,192
VAL_SPECS=[(11,2,1),(12,5,-2),(13,8,3),(14,-4,4)]
TEST_SPECS=[(21,3,2),(22,7,-3),(23,11,4),(24,-6,5)]
assert {s for s,dx,dy in VAL_SPECS}.isdisjoint({s for s,dx,dy in TEST_SPECS})

def make_pair(seed,dx,dy,kind):
 bg=texture(seed,H,W);x,y=grid(H,W);M=np.float32([[1,0,dx],[0,1,dy]])
 if kind=='translation':
  a=bg;b=cv2.warpAffine(a,M,(W,H),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_CONSTANT)
  gt=np.zeros((H,W,2),np.float32);gt[:]=[dx,dy]
  valid=(x+dx>=0)&(x+dx<=W-1)&(y+dy>=0)&(y+dy<=H-1)
  boundary=(x<8)|(x>=W-8)|(y<8)|(y>=H-8)
 else:
  obj=np.zeros((H,W),np.uint8);obj[42:86,66:126]=1
  foreground=texture(seed+1000,H,W);a=bg.copy();a[obj>0]=foreground[obj>0]
  moved_mask=cv2.warpAffine(obj,M,(W,H),flags=cv2.INTER_NEAREST)>0
  canvas=np.zeros_like(bg);canvas[obj>0]=foreground[obj>0];moved=cv2.warpAffine(canvas,M,(W,H),flags=cv2.INTER_NEAREST)
  b=bg.copy();b[moved_mask]=moved[moved_mask]
  gt=np.zeros((H,W,2),np.float32);gt[obj>0]=[dx,dy]
  # Background source covered by the moved foreground is not jointly visible.
  valid=((obj>0)|(~moved_mask))&(x+gt[...,0]>=0)&(x+gt[...,0]<=W-1)&(y+gt[...,1]>=0)&(y+gt[...,1]<=H-1)
  kernel=np.ones((7,7),np.uint8);boundary=(cv2.dilate(obj,kernel)>0)&(cv2.erode(obj,kernel)==0)
 gx=cv2.Sobel(a.astype(np.float32),cv2.CV_32F,1,0,ksize=3)/8;gy=cv2.Sobel(a.astype(np.float32),cv2.CV_32F,0,1,ksize=3)/8
 low_texture=np.hypot(gx,gy)<2
 name=f'{kind}_{seed}_{dx}_{dy}'
 digest=hashlib.sha256(a.tobytes()+b.tobytes()+gt.tobytes()+valid.tobytes()).hexdigest()
 return {'name':name,'seed':seed,'dx':dx,'dy':dy,'kind':kind,'a':a,'b':b,'gt':gt,'valid':valid,'boundary':boundary,'low_texture':low_texture,'sha256':digest}

val_pairs=[make_pair(seed,dx,dy,kind) for seed,dx,dy in VAL_SPECS for kind in ['translation','moving_patch']]
# Generator audit: exact GT reconstructs image1 on all jointly-visible pixels.
for pair in val_pairs:
 reconstructed,inside=warp_with_forward(pair['b'],pair['gt']);mask=pair['valid']&inside
 assert mask.sum()>0 and np.max(np.abs(reconstructed[mask]-pair['a'][mask]))==0
print('GT warp audit passed for',len(val_pairs),'validation pairs')

## Одинаковые алгоритмы и учет времени
Farneback: pyr_scale .5, levels 3, winsize 15, iterations 3, poly_n 5, poly_sigma 1.2. DIS: preset MEDIUM. Это классические оценщики, не нейросети. Один CPU thread фиксирован для воспроизводимого локального замера.

Время включает создание DIS-объекта, расчет flow и, для абляции, resize входов/возврат поля. Чтение данных, создание синтетики и рисунки исключены. Это локальный замер вызова; он не гарантирует real-time pipeline на другом оборудовании.

In [ ]:
METHOD_NAMES=['Farneback','DIS_medium']
FARNEBACK_CONFIG={'pyr_scale':.5,'levels':3,'winsize':15,'iterations':3,'poly_n':5,'poly_sigma':1.2,'flags':0}

def estimate(a,b,method,scale=1.):
 start=time.perf_counter();h,w=a.shape
 if scale!=1:
  size=(round(w*scale),round(h*scale));aa=cv2.resize(a,size,interpolation=cv2.INTER_AREA);bb=cv2.resize(b,size,interpolation=cv2.INTER_AREA)
 else:aa,bb=a,b
 if method=='Farneback':flow=cv2.calcOpticalFlowFarneback(aa,bb,None,**FARNEBACK_CONFIG)
 elif method=='DIS_medium':
  dis=cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_MEDIUM);flow=dis.calc(aa,bb,None)
 else:raise ValueError('unknown method')
 if scale!=1:flow=resize_flow(flow,(h,w))
 return flow,time.perf_counter()-start

def evaluate(pair,method,scale=1.):
 flow,seconds=estimate(pair['a'],pair['b'],method,scale);score=masked_epe(flow,pair['gt'],pair['valid'])
 boundary=masked_epe(flow,pair['gt'],pair['valid']&pair['boundary'])
 flat=masked_epe(flow,pair['gt'],pair['valid']&pair['low_texture'])
 row={'pair':pair['name'],'kind':pair['kind'],'method':method,'scale':scale,'epe':score['mean'],'valid_count':score['count'],'epe_sum':float(np.nansum(score['map'])),'boundary_epe':boundary['mean'],'low_texture_epe':flat['mean'],'seconds':seconds,'data_sha256':pair['sha256']}
 return row,flow,score['map']

## Выбор по val
Сначала проверяются оба метода на одинаковых восьми парах. Pooled EPE = сумма ошибок всех valid пикселей / общее число valid пикселей. В таблице также остаются EPE на границе и слабой текстуре; они нужны для диагностики, а не для смены критерия после результата.

In [ ]:
RESULTS=Path('results');RESULTS.mkdir(exist_ok=True)
def write_csv(name,rows):
 with (RESULTS/name).open('w',newline='') as f:
  writer=csv.DictWriter(f,fieldnames=rows[0]);writer.writeheader();writer.writerows(rows)
# One warmup on the same val pair per method; no test access.
for method in METHOD_NAMES:estimate(val_pairs[0]['a'],val_pairs[0]['b'],method)
val_rows=[evaluate(pair,method)[0] for method in METHOD_NAMES for pair in val_pairs]
summary=[]
for method in METHOD_NAMES:
 rows=[r for r in val_rows if r['method']==method]
 summary.append({'method':method,'pooled_epe':sum(r['epe_sum'] for r in rows)/sum(r['valid_count'] for r in rows),'valid_count':sum(r['valid_count'] for r in rows),'mean_seconds':float(np.mean([r['seconds'] for r in rows]))})
chosen=min(summary,key=lambda r:(r['pooled_epe'],METHOD_NAMES.index(r['method'])))
lock={'method_order':list(METHOD_NAMES),'selected_method':chosen['method'],'criterion':'min pooled GT-valid val EPE; ties method order','ablation_scale':.5,'farneback_config':FARNEBACK_CONFIG,'dis_preset':'MEDIUM','opencv':cv2.__version__,'cpu_threads':cv2.getNumThreads(),'val_specs':VAL_SPECS,'test_specs':TEST_SPECS,'GT_valid':'jointly visible source pixels; no predicted-mask filtering'}
(RESULTS/'selection_locked.json').write_text(json.dumps(lock,indent=2));write_csv('validation.csv',val_rows);write_csv('validation_summary.csv',summary)
print('VAL:',summary);print('LOCKED before generating test:',lock)
fig,axs=plt.subplots(1,2,figsize=(9,3),constrained_layout=True)
axs[0].bar([r['method'] for r in summary],[r['pooled_epe'] for r in summary]);axs[0].set(ylabel='pooled valid EPE (pixels)')
axs[1].bar([r['method'] for r in summary],[r['mean_seconds']*1000 for r in summary]);axs[1].set(ylabel='mean estimate pipeline (ms)')
fig.savefig(RESULTS/'validation.png');plt.show();plt.close(fig)

## Решение перед test
**Заполните:** выбранный метод, pooled EPE обоих, время, ожидаемый эффект scale .5 и вероятный сложный случай. Параметры записаны в `selection_locked.json`; далее test открывается с заранее объявленными тремя вариантами.

In [ ]:
lock=json.loads((RESULTS/'selection_locked.json').read_text())
# Validate the exact protocol before opening any test pair.
def require_locked_protocol(locked):
 current={'farneback_config':FARNEBACK_CONFIG,'dis_preset':'MEDIUM',
          'opencv':cv2.__version__,'cpu_threads':cv2.getNumThreads(),
          'val_specs':VAL_SPECS,'test_specs':TEST_SPECS,
          'method_order':METHOD_NAMES,'ablation_scale':.5}
 for key,value in current.items():
  if json.dumps(locked.get(key),sort_keys=True)!=json.dumps(value,sort_keys=True):
   raise RuntimeError('Протокол изменен после val: '+key+'. Вернитесь к val и зафиксируйте выбор заново до test.')
 if locked.get('selected_method') not in METHOD_NAMES:
  raise RuntimeError('В lock записан неизвестный метод.')
require_locked_protocol(lock)
test_pairs=[make_pair(seed,dx,dy,kind) for seed,dx,dy in TEST_SPECS for kind in ['translation','moving_patch']]
for pair in test_pairs:
 warped,inside=warp_with_forward(pair['b'],pair['gt']);mask=pair['valid']&inside
 assert np.max(np.abs(warped[mask]-pair['a'][mask]))==0
variants=[('Farneback','Farneback',1.),('DIS_medium','DIS_medium',1.),('selected_half',lock['selected_method'],lock['ablation_scale'])]
test_rows=[];predictions={};errors={}
for label,method,scale in variants:
 for pair in test_pairs:
  row,flow,error=evaluate(pair,method,scale);test_rows.append({'variant':label,**row});predictions[(label,pair['name'])]=flow;errors[(label,pair['name'])]=error
summary=[]
for label,method,scale in variants:
 rows=[r for r in test_rows if r['variant']==label]
 summary.append({'variant':label,'method':method,'scale':scale,'pooled_epe':sum(r['epe_sum'] for r in rows)/sum(r['valid_count'] for r in rows),'valid_count':sum(r['valid_count'] for r in rows),'mean_seconds':float(np.mean([r['seconds'] for r in rows]))})
write_csv('test.csv',test_rows);write_csv('test_summary.csv',summary)
for row in summary:print(row)
# Same first translation and first moving-patch pair; common color magnitude scale.
VMAX=12
fig,axs=plt.subplots(2,5,figsize=(13,5),constrained_layout=True)
for row,pair in enumerate(test_pairs[:2]):
 images=[pair['a'],pair['b'],flow_rgb(pair['gt'],VMAX),flow_rgb(predictions[('Farneback',pair['name'])],VMAX),flow_rgb(predictions[('DIS_medium',pair['name'])],VMAX)]
 for ax,img,title in zip(axs[row],images,['I1','I2','GT forward','Farneback','DIS medium']):
  ax.imshow(img,cmap='gray' if img.ndim==2 else None,vmin=0 if img.ndim==2 else None,vmax=255 if img.ndim==2 else None);ax.set_title(title);ax.axis('off')
 axs[row,0].set_title('I1 / '+pair['kind'])
fig.suptitle('Same inputs; flow brightness clipped at 12 pixels; direction legend shown above');fig.savefig(RESULTS/'same_pairs.png');plt.show();plt.close(fig)
# Failure diagnostic: predeclared largest-motion moving-patch test pair.
case=next(p for p in test_pairs if p['kind']=='moving_patch' and p['seed']==23)
fig,axs=plt.subplots(1,4,figsize=(12,3),constrained_layout=True)
axs[0].imshow(case['valid'],cmap='gray',vmin=0,vmax=1);axs[0].set_title('GT valid: white = included')
for ax,(label,method,scale) in zip(axs[1:],variants):
 im=ax.imshow(errors[(label,case['name'])],cmap=plt.get_cmap('magma').with_extremes(bad='lightgray'),vmin=0,vmax=8);ax.set_title(label+' EPE')
for ax in axs:ax.axis('off')
fig.colorbar(im,ax=axs[1:],label='EPE (pixels)',shrink=.7);fig.savefig(RESULTS/'failure_errors.png');plt.show();plt.close(fig)
# Forward/backward consistency is diagnostic, never used to change main EPE mask.
forward=predictions[(lock['selected_method'],case['name'])];backward,_=estimate(case['b'],case['a'],lock['selected_method'])
residual,inside,consistent=forward_backward_check(forward,backward,1.)
full=masked_epe(forward,case['gt'],case['valid']);selected=masked_epe(forward,case['gt'],case['valid']&consistent)
report={'pair':case['name'],'method':lock['selected_method'],'threshold_pixels':1.,'GT_valid_epe':full['mean'],'GT_valid_pixels':full['count'],'consistent_coverage_of_GT':float((case['valid']&consistent).sum()/case['valid'].sum()),'diagnostic_subset_epe':selected['mean'],'warning':'subset is method-selected; cannot replace common GT-valid primary score'}
(RESULTS/'consistency_diagnostic.json').write_text(json.dumps(report,indent=2));print(report)
np.savez_compressed(RESULTS/'failure_case.npz',image1=case['a'],image2=case['b'],gt=case['gt'],valid=case['valid'],forward=forward,backward=backward,consistency=residual)

## Реальная пара и добавленное движение изображения
Берем frames 100 и 101 OpenCV vtest и отдельно добавляем ко второму реальному кадру перенос (+3,+1). Истинное движение исходной реальной пары неизвестно, поэтому перенос второго кадра **не делает полное поле GT известным**.

Сравните стрелки двух методов на одинаковой сетке I1. Для читаемости все стрелки увеличены в 4 раза; отметка 2 px показывает настоящий модуль потока. Объясните, где доминирует добавленное движение всего изображения и где видны локальные отличия. Median flow — грубая описательная статистика, не гарантированная оценка движения камеры. Photometric MAE — residual изображений, не EPE; он чувствителен к освещению, перекрытиям и текстуре.

Реальные данные не участвуют в выборе метода или scale. Вне GT нельзя делать вывод о победителе только по красивой цветовой карте или меньшему residual.

In [ ]:
import urllib.request,tempfile
VIDEO_URL='https://raw.githubusercontent.com/opencv/opencv/4.10.0/samples/data/vtest.avi'
VIDEO_SHA='45cddc9490be69345cbdab64ca583be65987e864ca408038e648db99e10516cf'
cache=Path(tempfile.gettempdir())/'applied-ai-tracking';cache.mkdir(exist_ok=True)
video_path=Path(os.environ.get('COURSE_VIDEO_PATH',cache/'vtest.avi'))
if not video_path.exists():
 with urllib.request.urlopen(VIDEO_URL,timeout=60) as response:video_path.write_bytes(response.read())
assert hashlib.sha256(video_path.read_bytes()).hexdigest()==VIDEO_SHA
cap=cv2.VideoCapture(str(video_path));assert cap.isOpened();cap.set(cv2.CAP_PROP_POS_FRAMES,100)
ok,a_rgb=cap.read();assert ok;ok,b_rgb=cap.read();assert ok;cap.release()
a_rgb=cv2.resize(a_rgb,(384,288));b_rgb=cv2.resize(b_rgb,(384,288));a=cv2.cvtColor(a_rgb,cv2.COLOR_BGR2GRAY);b=cv2.cvtColor(b_rgb,cv2.COLOR_BGR2GRAY)
# Artificial (+3,+1) translation is added to the REAL second frame, not to I1.
# Underlying real motion remains unknown; this is NOT full flow ground truth.
b_aug=cv2.warpAffine(b,np.float32([[1,0,3],[0,1,1]]),(384,288))
real_rows=[];real_flows={}
for name,image2 in [('real_pair',b),('real_plus_translation',b_aug)]:
 for method in METHOD_NAMES:
  flow,seconds=estimate(a,image2,method);recon,inside=warp_with_forward(image2,flow)
  # Conservative fixed image border to avoid augmented black border; no GT inference.
  area=inside.copy();area[:8]=False;area[-8:]=False;area[:,:8]=False;area[:,-8:]=False
  mae=float(np.abs(recon-a)[area].mean());median=np.median(flow[area],axis=0)
  real_rows.append({'case':name,'method':method,'seconds':seconds,'photometric_MAE_0_255':mae,'median_u':float(median[0]),'median_v':float(median[1]),'ground_truth':False});real_flows[(name,method)]=flow
write_csv('real_qualitative.csv',real_rows)
fig,axs=plt.subplots(2,3,figsize=(11,6),constrained_layout=True)
for row,(name,image2) in enumerate([('real_pair',b),('real_plus_translation',b_aug)]):
 axs[row,0].imshow(image2,cmap='gray',vmin=0,vmax=255);axs[row,0].set_title(name+' / I2')
 for col,method in enumerate(METHOD_NAMES,1):
  flow=real_flows[(name,method)];axs[row,col].imshow(a,cmap='gray',vmin=0,vmax=255)
  y,x=np.mgrid[12:288:24,12:384:24];q=axs[row,col].quiver(x,y,flow[y,x,0],flow[y,x,1],angles='xy',scale_units='xy',scale=.25,color='tomato',width=.003)
  axs[row,col].quiverkey(q,.78,.96,2.,'2 px',labelpos='E',coordinates='axes',color='yellow')
  axs[row,col].set_title(method+' / forward arrows ×4')
 for ax in axs[row]:ax.axis('off')
fig.suptitle('Arrows on I1, common display factor ×4; real motion has no GT');fig.savefig(RESULTS/'real_flow.png');plt.show();plt.close(fig)
(RESULTS/'real_video_run.json').write_text(json.dumps({'source_sha256':VIDEO_SHA,'source_frames':[100,101],'resize':[384,288],'second_frame_added_translation':[3,1],'ground_truth':False,'opencv':cv2.__version__},indent=2))
for row in real_rows:print(row)

## Сдача и выводы
1. Таблица минимум 18 опытов демо: по два на каждый из девяти стендов, реализованные TODO, собственные проверки.
2. Val/test CSV, fixed config, графики одинаковых пар и error map. Сопоставьте время с EPE в исходных пикселях.
3. Разберите один чистый перенос и одну границу движения на тех же парах у обоих методов. Назовите конкретную пару и ее boundary/low-texture EPE.
4. Объясните consistency diagnostic: сколько GT-valid пикселей осталось, как изменилось среднее на выбранном методом подмножестве и почему этот score нельзя подставлять в общую таблицу.
5. Три наблюдения на реальной паре, включая добавленный перенос; явно укажите отсутствие GT. Минимум два ограничения синтетики и один следующий проверяемый опыт.

Не перебирать новые параметры по test. Дополнительный опыт можно провести на val с заранее записанным прогнозом и отдельной маркировкой.

Источники: [OpenCV optical flow](https://docs.opencv.org/4.x/d4/dee/tutorial_optical_flow.html), [DIS](https://arxiv.org/abs/1603.03590), [Farneback](https://www.ida.liu.se/ext/WITAS-ev/Computer_Vision_Technologies/Papers/scia03_farneback.pdf).